<a href="https://colab.research.google.com/github/xxmadara1xx/Controllo-LED-tramite-UART/blob/main/drone_best_pt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚁 Drone Detection - النسخة النهائية
### الخطوات:
1. شغّل **الخلية 1** ثم اضغط `Runtime` ← `Restart runtime`
2. شغّل بقية الخلايا بالترتيب
3. غير أسماء ملفاتك في **الخلية 4** فقط

In [ ]:
# ==============================
# الخلية 1: تثبيت وإصلاح
# بعد انتهاء هذه الخلية اضغط:
# Runtime → Restart runtime
# ==============================
!git clone https://github.com/ultralytics/yolov5 /content/yolov5 -q
!pip install -r /content/yolov5/requirements.txt -q
!pip install opencv-python-headless Pillow -q

# إصلاح مشكلة PyTorch 2.6
import re

for fpath in [
    '/content/yolov5/models/experimental.py',
    '/content/yolov5/utils/general.py',
]:
    with open(fpath, 'r') as f:
        content = f.read()
    fixed = re.sub(
        r"torch\.load\(([^)]*?)\)",
        lambda m: m.group(0) if 'weights_only' in m.group(0)
                  else m.group(0)[:-1] + ', weights_only=False)',
        content
    )
    with open(fpath, 'w') as f:
        f.write(fixed)
    print(f'✅ تم إصلاح: {fpath.split("/")[-1]}')

print('\n✅ اكتمل التثبيت!')
print('⚠️  الآن اضغط: Runtime → Restart runtime')
print('⚠️  ثم شغّل من الخلية 2')

fatal: destination path '/content/yolov5' already exists and is not an empty directory.
✅ تم إصلاح: experimental.py
✅ تم إصلاح: general.py

✅ اكتمل التثبيت!
⚠️  الآن اضغط: Runtime → Restart runtime
⚠️  ثم شغّل من الخلية 2


In [ ]:
# ==============================
# الخلية 2: استيراد المكتبات
# ==============================
import sys
sys.path.insert(0, '/content/yolov5')

import torch
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from base64 import b64encode
import os, shutil

from models.experimental import attempt_load
from utils.general import non_max_suppression, scale_boxes
from utils.augmentations import letterbox

print('✅ تم الاستيراد!')

✅ تم الاستيراد!


In [ ]:
# ==============================
# الخلية 3: ربط Google Drive
# ==============================
from google.colab import drive
drive.mount('/content/drive')
print('\n📂 ملفاتك في Drive:')
!ls /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📂 ملفاتك في Drive:
 aeroporto.jpg						   result_image.jpg
 best.pt						   result_video.mp4
 best_unified.pt					   testdrone1.jpg
'Colab Notebooks'					   testdrone2.jpg
'Copie de PICT Comp TE Seminar Report | Docx2LaTeX.gdoc'   testdrone3.jpg
 drone1.mp4						   testdrone5.jpeg
 drone.mp4						   testdrone.jpg
'Google AI Studio'					   uavmult.mp4
 military-drone-detection.v19i.yolov5pytorch.zip


In [50]:
# ==============================
# الخلية 4: ✏️ غير أسماء ملفاتك هنا
# ==============================
MODEL_NAME = 'best_unified.pt'    # ← اسم نموذجك في Drive
IMAGE_NAME = 'aeroporto.jpg'  # ← اسم صورتك في Drive
VIDEO_NAME = 'uavmult.mp4'  # ← اسم فيديوك في Drive
MAX_FRAMES = 999999          # ← عدد إطارات الفيديو
CONF       = 0.3          # ← دقة الكشف (0.1-0.9)
IOU        = 0.6         # ← تجنب التكرار

MODEL_PATH = f'/content/drive/MyDrive/{MODEL_NAME}'
IMAGE_PATH = f'/content/drive/MyDrive/{IMAGE_NAME}'
VIDEO_PATH = f'/content/drive/MyDrive/{VIDEO_NAME}'

print('📋 الإعدادات:')
print(f'  النموذج : {"✅" if os.path.exists(MODEL_PATH) else "❌"} {MODEL_NAME}')
print(f'  الصورة  : {"✅" if os.path.exists(IMAGE_PATH) else "❌"} {IMAGE_NAME}')
print(f'  الفيديو : {"✅" if os.path.exists(VIDEO_PATH) else "❌"} {VIDEO_NAME}')

📋 الإعدادات:
  النموذج : ✅ best_unified.pt
  الصورة  : ✅ aeroporto.jpg
  الفيديو : ✅ uavmult.mp4


In [51]:
# ==============================
# الخلية 5: تحميل النموذج
# ==============================
if os.path.exists(MODEL_PATH):
    model = attempt_load(MODEL_PATH, device='cpu')
    print(f'✅ تم تحميل نموذجك: {MODEL_NAME}')
else:
    print(f'❌ {MODEL_NAME} غير موجود في Drive!')
    print('📂 الملفات الموجودة:')
    !ls /content/drive/MyDrive/*.pt
    raise FileNotFoundError(f'ارفع {MODEL_NAME} على Drive أولاً')

model.eval()
NAMES = model.names if hasattr(model, 'names') else {i: str(i) for i in range(100)}
print(f'📋 الفئات: {NAMES}')

Fusing layers... 
Model summary: 157 layers, 7023610 parameters, 0 gradients, 15.8 GFLOPs


✅ تم تحميل نموذجك: best_unified.pt
📋 الفئات: {0: 'Helicopter', 1: 'UAV', 2: 'airplane', 3: 'birds', 4: 'drone'}


In [55]:

# ==============================
# الخلية 6: دوال الكشف
# ==============================
def detect(frame_bgr, img_size=640):
    img0 = frame_bgr.copy()
    img  = letterbox(img0, img_size, stride=32, auto=True)[0]
    img  = img.transpose((2, 0, 1))[::-1]
    img  = np.ascontiguousarray(img)
    img  = torch.from_numpy(img).float() / 255.0
    img  = img.unsqueeze(0)
    with torch.no_grad():
        pred = model(img)[0]
    pred = non_max_suppression(pred, CONF, IOU,
                           agnostic=False,  # ← غير هذا
                           max_det=100)
    return pred, img.shape[2:]

def draw_boxes(frame_bgr, pred, img_shape):
    count = 0
    h, w  = frame_bgr.shape[:2]
    frame_area = h * w

    for det in pred:
        if len(det):
            det[:, :4] = scale_boxes(img_shape, det[:, :4], frame_bgr.shape).round()
            for *xyxy, conf, cls in det:
                x1, y1, x2, y2 = map(int, xyxy)

                # تجاهل المربعات الكبيرة جداً
                box_area = (x2 - x1) * (y2 - y1)
                if box_area > frame_area * 0.6:
                    continue

                count += 1
                # مربع الكشف
                cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 0, 255), 3)
                # الاسم والثقة
                label = f'{NAMES[int(cls)]}: {conf*100:.0f}%'
                cv2.putText(frame_bgr, label, (x1, max(y1-10, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                # الإحداثيات
                cx, cy = (x1+x2)//2, (y1+y2)//2
                cv2.putText(frame_bgr, f'({cx},{cy})', (x1, y2+25),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return frame_bgr, count

print('✅ تم تعريف دوال الكشف!')

✅ تم تعريف دوال الكشف!


In [58]:
# ==============================
# الخلية 7: الكشف على صورة
# ==============================
if not os.path.exists(IMAGE_PATH):
    print(f'❌ الصورة غير موجودة: {IMAGE_NAME}')
    print('📂 الملفات المتاحة:')
    !ls /content/drive/MyDrive/
else:
    frame = cv2.imread(IMAGE_PATH)
    print(f'✅ تم تحميل الصورة: {frame.shape}')

    pred, img_shape = detect(frame)
    frame, count    = draw_boxes(frame, pred, img_shape)

    # حفظ في Drive
    out_path = '/content/drive/MyDrive/result_image.jpg'
    cv2.imwrite(out_path, frame)

    # عرض
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(f'🚁 تم الكشف عن {count} طائرة', fontsize=16)
    plt.axis('off')
    plt.show()
    print(f'✅ عدد الطائرات: {count}')
    print(f'💾 محفوظة في Drive: result_image.jpg')

✅ تم تحميل الصورة: (620, 950, 3)
✅ عدد الطائرات: 2
💾 محفوظة في Drive: result_image.jpg


In [59]:
# ==============================
# الخلية 8: الكشف على فيديو
# ==============================
if not os.path.exists(VIDEO_PATH):
    print(f'❌ الفيديو غير موجود: {VIDEO_NAME}')
    print('📂 الملفات المتاحة:')
    !ls /content/drive/MyDrive/
else:
    print('⏳ نسخ الفيديو محلياً...')
    shutil.copy(VIDEO_PATH, '/content/input.mp4')

    cap    = cv2.VideoCapture('/content/input.mp4')
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = max(int(cap.get(cv2.CAP_PROP_FPS)), 25)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'📹 {width}x{height} @ {fps}fps | {total} إطار')
    print(f'⏳ معالجة {min(MAX_FRAMES, total)} إطار...')

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter('/content/output_raw.mp4', fourcc, fps, (width, height))

    frame_count  = 0
    drone_frames = 0

    while cap.isOpened() and frame_count < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break

        pred, img_shape = detect(frame, img_size=416)
        frame, count    = draw_boxes(frame, pred, img_shape)

        if count > 0:
            drone_frames += 1
            cv2.putText(frame, f'⚠ DRONE DETECTED! ({count})', (20, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)

        cv2.putText(frame, f'Frame:{frame_count}/{min(MAX_FRAMES,total)}',
                    (20, height-20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

        out.write(frame)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f'  ⏳ {frame_count}/{min(MAX_FRAMES,total)} إطار | طائرات: {drone_frames}')

    cap.release()
    out.release()

    # تحويل الفيديو
    print('\n⏳ تحويل الفيديو...')
    !ffmpeg -i /content/output_raw.mp4 -vcodec libx264 /content/output_final.mp4 -y -loglevel quiet

    # حفظ في Drive
    shutil.copy('/content/output_final.mp4', '/content/drive/MyDrive/result_video.mp4')

    print(f'\n✅ اكتمل!')
    print(f'🚁 إطارات تحتوي طائرة: {drone_frames}/{frame_count} ({drone_frames/max(frame_count,1)*100:.1f}%)')
    print(f'💾 محفوظ في Drive: result_video.mp4')

    # عرض الفيديو
    mp4      = open('/content/output_final.mp4', 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    display(HTML(f'<video width=640 controls autoplay><source src="{data_url}" type="video/mp4"></video>'))

⏳ نسخ الفيديو محلياً...
📹 768x432 @ 29fps | 630 إطار
⏳ معالجة 630 إطار...
  ⏳ 30/630 إطار | طائرات: 24
  ⏳ 60/630 إطار | طائرات: 39
  ⏳ 90/630 إطار | طائرات: 44
  ⏳ 120/630 إطار | طائرات: 57
  ⏳ 150/630 إطار | طائرات: 70
  ⏳ 180/630 إطار | طائرات: 75
  ⏳ 210/630 إطار | طائرات: 76
  ⏳ 240/630 إطار | طائرات: 78
  ⏳ 270/630 إطار | طائرات: 78
  ⏳ 300/630 إطار | طائرات: 78
  ⏳ 330/630 إطار | طائرات: 87
  ⏳ 360/630 إطار | طائرات: 91
  ⏳ 390/630 إطار | طائرات: 99
  ⏳ 420/630 إطار | طائرات: 129
  ⏳ 450/630 إطار | طائرات: 159
  ⏳ 480/630 إطار | طائرات: 189
  ⏳ 510/630 إطار | طائرات: 219
  ⏳ 540/630 إطار | طائرات: 249
  ⏳ 570/630 إطار | طائرات: 265
  ⏳ 600/630 إطار | طائرات: 265
  ⏳ 630/630 إطار | طائرات: 265

⏳ تحويل الفيديو...

✅ اكتمل!
🚁 إطارات تحتوي طائرة: 265/630 (42.1%)
💾 محفوظ في Drive: result_video.mp4
